In [36]:
import os
import json
import random

def calculate_penalty(proposed_group1, proposed_group2, history):
    penalty = 0
    for idx, past_meeting in enumerate(history):
        weight = 1.0 / (idx + 1) 
        past_groups = past_meeting['groups']     # [0]은 과거 1조, [1]은 과거 2조
        past_g1, past_g2 = past_groups[0], past_groups[1]
        
        # --- 1. 사람 간의 중복 편성 벌점 (기존 로직) ---
        for proposed_group in [proposed_group1, proposed_group2]:
            for i in range(len(proposed_group)):
                for j in range(i + 1, len(proposed_group)):
                    p1, p2 = proposed_group[i], proposed_group[j]
                    if (p1 in past_g1 and p2 in past_g1) or (p1 in past_g2 and p2 in past_g2):
                        penalty += weight
                        
        # --- 2. 특정 조 고착화 방지 벌점 (추가된 로직) ---
        # 새로 제안된 1조(proposed_group1)의 사람이 과거에도 1조(past_g1)였으면 벌점
        for p in proposed_group1:
            if p in past_g1:
                penalty += weight * 0.5  # 조 고착 벌점은 사람 중복보다는 낮게 가중치 설정 (0.5 배)
                
        # 새로 제안된 2조(proposed_group2)의 사람이 과거에도 2조(past_g2)였으면 벌점
        for p in proposed_group2:
            if p in past_g2:
                penalty += weight * 0.5
                
    return penalty

def divide_groups(attendees, meeting_date_str, folder_path="history"):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        
    history = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                history.append({
                    'date': filename.replace('.json', ''),
                    'groups': [data.get('group1', []), data.get('group2', [])]
                })
                
    history.sort(key=lambda x: x['date'], reverse=True)
    
    target_person = "오정환"
    has_target = target_person in attendees
    
    pool = attendees.copy()
    if has_target:
        pool.remove(target_person)
        
    size1 = len(pool) // 2
    
    best_division = None
    min_penalty = float('inf')
    
    for _ in range(1000):
        random.shuffle(pool)
        
        if has_target:
            g1 = pool[:size1]
            g2 = pool[size1:] + [target_person]
        else:
            g1 = pool[:len(pool)//2]
            g2 = pool[len(pool)//2:]
        
        penalty = calculate_penalty(g1, g2, history)
        
        if best_division is None or penalty < min_penalty:
            min_penalty = penalty
            best_division = (list(g1), list(g2))
            
        if penalty == 0:
            break
            
    final_g1, final_g2 = best_division
    
    safe_date_str = meeting_date_str.replace('/', '-')
    filepath = os.path.join(folder_path, f"{safe_date_str}.json")
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump({"group1": final_g1, "group2": final_g2}, f, ensure_ascii=False, indent=2)
        
    return final_g1, final_g2

# --- 실행 부분 ---
today_attendees = ["구교진", "김택민", "이동건", "박찬하", "현지용", "장하연"]
today_date = "2026/06/21"

group1, group2 = divide_groups(today_attendees, today_date, './history')

print(f"[{today_date} 목장 모임 조 편성 결과]")
print(f"1조 ({len(group1)}명): {', '.join(group1)}")
print(f"2조 ({len(group2)}명): {', '.join(group2)}")

[2026/06/21 목장 모임 조 편성 결과]
1조 (3명): 구교진, 김택민, 장하연
2조 (3명): 현지용, 이동건, 박찬하
